In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [5]:
!pip install transformers datasets accelerate -q

# **Load Data**

In [6]:
from datasets import load_dataset

dataset = load_dataset("roneneldan/TinyStories") # a dataset from hugging face that's specifically for training small language models (Same as BabyLM dataset). has pre made sections train and validation sections
print(dataset)
print(dataset["train"][0])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})
{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


# **Tokenizer**

In [7]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT2 has no pad token by default

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

# Use a SMALL subset first so everything runs fast while you debug
small_train = dataset["train"].select(range(20000))
small_val = dataset["validation"].select(range(2000))

tokenized_train = small_train.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_val = small_val.map(tokenize_fn, batched=True, remove_columns=["text"])

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

# **Data collator (handles batching/padding)**

In [8]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# **Teacher Model(Alerady Trained)**

In [9]:
from transformers import GPT2LMHeadModel

teacher = GPT2LMHeadModel.from_pretrained("gpt2")  # 124M params
teacher.eval()  # teacher doesn't train, just gives predictions
teacher.to("cuda")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

# **Student Model**

In [11]:
from transformers import GPT2Config, GPT2LMHeadModel

student_config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=128,
    n_embd=128,
    n_layer=4,
    n_head=4,
)
student = GPT2LMHeadModel(student_config)
student.to("cuda")

print(f"Student params: {sum(p.numel() for p in student.parameters())/1e6:.1f}M")

Student params: 7.2M


# **Baseline #1 (Training student without distillation)**

In [12]:
from transformers import Trainer, TrainingArguments

baseline_args = TrainingArguments(
    output_dir="./baseline_student",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="no",
    report_to="none",
)

baseline_trainer = Trainer(
    model=student,
    args=baseline_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

baseline_trainer.train()
baseline_eval = baseline_trainer.evaluate()
print("Baseline (no distillation) eval loss:", baseline_eval["eval_loss"])

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
200,8.752911,8.504660
400,7.401763,7.231064
600,6.620001,6.517742
800,6.222626,6.145816
1000,6.007780,5.956182
1200,5.931612,5.883845


Baseline (no distillation) eval loss: 5.880754470825195


# **Training student with distillation**

In [13]:
# Fresh student for the distillation run (same architecture, brand new random weights)
student_distilled = GPT2LMHeadModel(student_config)
student_distilled.to("cuda")

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 128)
    (wpe): Embedding(128, 128)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-3): 4 x GPT2Block(
        (ln_1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=384, nx=128)
          (c_proj): Conv1D(nf=128, nx=128)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=512, nx=128)
          (c_proj): Conv1D(nf=128, nx=512)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=128, out_features=50257, bias=False)
)

# **Custom Trainer: Combining Real-Answer Loss + Teacher-Matching Loss**

In [19]:
import torch
import torch.nn.functional as F
from transformers import Trainer

class DistillationTrainer(Trainer):
    def __init__(self, teacher_model=None, temperature=2.0, alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            outputs = model(**inputs)
            student_loss = outputs.loss
            student_logits = outputs.logits
            teacher_logits_shape = student_logits.shape

            with torch.no_grad():
                teacher_outputs = self.teacher(**inputs)
                teacher_logits = teacher_outputs.logits

            T = self.temperature

            # Flatten (batch, seq_len, vocab) -> (batch*seq_len, vocab)
            # so each TOKEN is treated as one "example", giving a proper average
            student_logits_flat = student_logits.view(-1, student_logits.size(-1))
            teacher_logits_flat = teacher_logits.view(-1, teacher_logits.size(-1))

            distill_loss = F.kl_div(
                F.log_softmax(student_logits_flat / T, dim=-1),
                F.softmax(teacher_logits_flat / T, dim=-1),
                reduction="batchmean"
            ) * (T ** 2)

            loss = self.alpha * student_loss + (1 - self.alpha) * distill_loss

            return (loss, outputs) if return_outputs else loss

# **Train Student WITH Distillation (Tiny Size)**

In [ ]:
distill_args = TrainingArguments(
    output_dir="./distilled_student",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="no",
    report_to="none",
)

distill_trainer = DistillationTrainer(
    model=student_distilled,
    teacher_model=teacher,
    temperature=2.0,
    alpha=0.5,
    args=distill_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

distill_trainer.train()
distill_eval = distill_trainer.evaluate()
print("Distilled student eval loss:", distill_eval["eval_loss"])

# **Small: Build Student (small config)**

In [14]:
small_config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=128,
    n_embd=192,
    n_layer=6,
    n_head=6,
)

student_small_baseline = GPT2LMHeadModel(small_config)
student_small_baseline.to("cuda")

print(f"Small student params: {sum(p.numel() for p in student_small_baseline.parameters())/1e6:.1f}M")

Small student params: 12.3M


# **Small: Baseline Training (No Distillation)**

In [ ]:
small_args = TrainingArguments(
    output_dir="./small_baseline",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="no",
    report_to="none",
)

small_baseline_trainer = Trainer(
    model=student_small_baseline,
    args=small_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

small_baseline_trainer.train()
small_baseline_eval = small_baseline_trainer.evaluate()
print("Small baseline eval loss:", small_baseline_eval["eval_loss"])

# **Small: Distilled Training**

In [ ]:
student_small_distilled = GPT2LMHeadModel(small_config)
student_small_distilled.to("cuda")

small_distill_trainer = DistillationTrainer(
    model=student_small_distilled,
    teacher_model=teacher,
    temperature=2.0,
    alpha=0.5,
    args=small_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

small_distill_trainer.train()
small_distill_eval = small_distill_trainer.evaluate()
print("Small distilled eval loss:", small_distill_eval["eval_loss"])

# **Results So Far**

In [ ]:
print("=== TINY (4 layers, 128 dim) ===")
print(f"Baseline loss:  5.86")
print(f"Distilled loss: 4.52")
print(f"Gap: {5.86 - 4.52:.2f}")
print()
print("=== SMALL (6 layers, 192 dim) ===")
print(f"Baseline loss:  {small_baseline_eval['eval_loss']:.2f}")
print(f"Distilled loss: {small_distill_eval['eval_loss']:.2f}")
print(f"Gap: {small_baseline_eval['eval_loss'] - small_distill_eval['eval_loss']:.2f}")

# **Medium: Build Student (Medium config)**

In [15]:
medium_config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=128,
    n_embd=256,
    n_layer=8,
    n_head=8,
)

student_medium_baseline = GPT2LMHeadModel(medium_config)
student_medium_baseline.to("cuda")

print(f"Medium student params: {sum(p.numel() for p in student_medium_baseline.parameters())/1e6:.1f}M")

Medium student params: 19.2M


# **Medium: Baseline Training (No Distillation)**

In [ ]:
medium_args = TrainingArguments(
    output_dir="./medium_baseline",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="no",
    report_to="none",
)

medium_baseline_trainer = Trainer(
    model=student_medium_baseline,
    args=medium_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

medium_baseline_trainer.train()
medium_baseline_eval = medium_baseline_trainer.evaluate()
print("Medium baseline eval loss:", medium_baseline_eval["eval_loss"])

# **Medium: Distilled Training**

In [ ]:
student_medium_distilled = GPT2LMHeadModel(medium_config)
student_medium_distilled.to("cuda")

medium_distill_trainer = DistillationTrainer(
    model=student_medium_distilled,
    teacher_model=teacher,
    temperature=2.0,
    alpha=0.5,
    args=medium_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

medium_distill_trainer.train()
medium_distill_eval = medium_distill_trainer.evaluate()
print("Medium distilled eval loss:", medium_distill_eval["eval_loss"])

# **Results So Far (All Sizes)**

In [ ]:
results = [
    ("Tiny (4L, 128d)", 5.86, 4.52),
    ("Small (6L, 192d)", small_baseline_eval["eval_loss"], small_distill_eval["eval_loss"]),
    ("Medium (8L, 256d)", medium_baseline_eval["eval_loss"], medium_distill_eval["eval_loss"]),
]

print(f"{'Size':<20}{'Baseline':<12}{'Distilled':<12}{'Gap':<10}")
for name, base, dist in results:
    gap = base - dist
    print(f"{name:<20}{base:<12.2f}{dist:<12.2f}{gap:<10.2f}")

# **Small v2**

In [ ]:
import torch
torch.manual_seed(456)

student_small_baseline_v2 = GPT2LMHeadModel(small_config)
student_small_baseline_v2.to("cuda")

# **Small: baseline v2**

In [ ]:
small_baseline_trainer_v2 = Trainer(
    model=student_small_baseline_v2,
    args=small_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

small_baseline_trainer_v2.train()
small_baseline_eval_v2 = small_baseline_trainer_v2.evaluate()
print("Small baseline v2 eval loss:", small_baseline_eval_v2["eval_loss"])

# **Small: distilled v2**

In [ ]:
torch.manual_seed(456)

student_small_distilled_v2 = GPT2LMHeadModel(small_config)
student_small_distilled_v2.to("cuda")

small_distill_trainer_v2 = DistillationTrainer(
    model=student_small_distilled_v2,
    teacher_model=teacher,
    temperature=2.0,
    alpha=0.5,
    args=small_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

small_distill_trainer_v2.train()
small_distill_eval_v2 = small_distill_trainer_v2.evaluate()
print("Small distilled v2 eval loss:", small_distill_eval_v2["eval_loss"])

In [ ]:
temp_small = GPT2LMHeadModel(small_config)
print(f"Small params: {sum(p.numel() for p in temp_small.parameters())/1e6:.2f}M")
del temp_small

temp_medium = GPT2LMHeadModel(medium_config)
print(f"Medium params: {sum(p.numel() for p in temp_medium.parameters())/1e6:.2f}M")
del temp_medium

# **Medium v2**

In [ ]:
torch.manual_seed(456)

student_medium_baseline_v2 = GPT2LMHeadModel(medium_config)
student_medium_baseline_v2.to("cuda")

# **Medium baseline v2**

In [ ]:
medium_baseline_trainer_v2 = Trainer(
    model=student_medium_baseline_v2,
    args=medium_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

medium_baseline_trainer_v2.train()
medium_baseline_eval_v2 = medium_baseline_trainer_v2.evaluate()
print("Medium baseline v2 eval loss:", medium_baseline_eval_v2["eval_loss"])

# **Medium distilled v2**

In [ ]:
torch.manual_seed(456)

student_medium_distilled_v2 = GPT2LMHeadModel(medium_config)
student_medium_distilled_v2.to("cuda")

medium_distill_trainer_v2 = DistillationTrainer(
    model=student_medium_distilled_v2,
    teacher_model=teacher,
    temperature=2.0,
    alpha=0.5,
    args=medium_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

medium_distill_trainer_v2.train()
medium_distill_eval_v2 = medium_distill_trainer_v2.evaluate()
print("Medium distilled v2 eval loss:", medium_distill_eval_v2["eval_loss"])

# **Get exact parameter counts for small and medium (tiny config)**

In [16]:
temp_small = GPT2LMHeadModel(small_config)
small_params = sum(p.numel() for p in temp_small.parameters())/1e6
print(f"Small params: {small_params:.2f}M")
del temp_small

temp_medium = GPT2LMHeadModel(medium_config)
medium_params = sum(p.numel() for p in temp_medium.parameters())/1e6
print(f"Medium params: {medium_params:.2f}M")
del temp_medium

# Also recompute Tiny's exact count for consistency
tiny_config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=128,
    n_embd=128,
    n_layer=4,
    n_head=4,
)
temp_tiny = GPT2LMHeadModel(tiny_config)
tiny_params = sum(p.numel() for p in temp_tiny.parameters())/1e6
print(f"Tiny params: {tiny_params:.2f}M")
del temp_tiny

Small params: 12.34M
Medium params: 19.22M
Tiny params: 7.24M


# **Visualization**

In [ ]:
import matplotlib.pyplot as plt

# Fill these in with your actual numbers once you have them
sizes_params = [tiny_params, small_params, medium_params]
gaps = [1.34, (0.089+0.096)/2, (0.194+0.210)/2]  # averaged across your 2 runs

plt.figure(figsize=(7,5))
plt.plot(sizes_params, gaps, marker='o', linewidth=2)
plt.xlabel("Student model size (million parameters)")
plt.ylabel("Distillation benefit (baseline loss − distilled loss)")
plt.title("Distillation Benefit vs. Student Model Size")
plt.grid(True, alpha=0.3)
plt.savefig("distillation_gap_vs_size.png", dpi=150, bbox_inches="tight")
plt.show()

# **"XSmall" (between Tiny and Small) student**

In [22]:
xsmall_config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=128,
    n_embd=160,
    n_layer=4,
    n_head=4,
)

temp_xsmall = GPT2LMHeadModel(xsmall_config)
xsmall_params = sum(p.numel() for p in temp_xsmall.parameters())/1e6
print(f"XSmall params: {xsmall_params:.2f}M")
del temp_xsmall

XSmall params: 9.30M


# **XSmall Baseline Training**

In [ ]:
torch.manual_seed(42)

student_xsmall_baseline = GPT2LMHeadModel(xsmall_config)
student_xsmall_baseline.to("cuda")

xsmall_args = TrainingArguments(
    output_dir="./xsmall_baseline",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="no",
    report_to="none",
)

xsmall_baseline_trainer = Trainer(
    model=student_xsmall_baseline,
    args=xsmall_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

xsmall_baseline_trainer.train()
xsmall_baseline_eval = xsmall_baseline_trainer.evaluate()
print("XSmall baseline eval loss:", xsmall_baseline_eval["eval_loss"])

# **XSmall Distilled Training**

In [ ]:
torch.manual_seed(42)

student_xsmall_distilled = GPT2LMHeadModel(xsmall_config)
student_xsmall_distilled.to("cuda")

xsmall_distill_trainer = DistillationTrainer(
    model=student_xsmall_distilled,
    teacher_model=teacher,
    temperature=2.0,
    alpha=0.5,
    args=xsmall_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

xsmall_distill_trainer.train()
xsmall_distill_eval = xsmall_distill_trainer.evaluate()
print("XSmall distilled eval loss:", xsmall_distill_eval["eval_loss"])

# **XSmall baseline v2**

In [ ]:
torch.manual_seed(789)

student_xsmall_baseline_v2 = GPT2LMHeadModel(xsmall_config)
student_xsmall_baseline_v2.to("cuda")

xsmall_baseline_trainer_v2 = Trainer(
    model=student_xsmall_baseline_v2,
    args=xsmall_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

xsmall_baseline_trainer_v2.train()
xsmall_baseline_eval_v2 = xsmall_baseline_trainer_v2.evaluate()
print("XSmall baseline v2 eval loss:", xsmall_baseline_eval_v2["eval_loss"])

# **XSmall distilled v2**

In [ ]:
torch.manual_seed(789)

student_xsmall_distilled_v2 = GPT2LMHeadModel(xsmall_config)
student_xsmall_distilled_v2.to("cuda")

xsmall_distill_trainer_v2 = DistillationTrainer(
    model=student_xsmall_distilled_v2,
    teacher_model=teacher,
    temperature=2.0,
    alpha=0.5,
    args=xsmall_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

xsmall_distill_trainer_v2.train()
xsmall_distill_eval_v2 = xsmall_distill_trainer_v2.evaluate()
print("XSmall distilled v2 eval loss:", xsmall_distill_eval_v2["eval_loss"])

# **"zoom-in" size at ~8M parameters (between Tiny 7.24M and XSmall 9.3M) - [zoom config]**

In [17]:
zoom_config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=128,
    n_embd=144,
    n_layer=4,
    n_head=4,
)

temp_zoom = GPT2LMHeadModel(zoom_config)
zoom_params = sum(p.numel() for p in temp_zoom.parameters())/1e6
print(f"Zoom params: {zoom_params:.2f}M")
del temp_zoom

Zoom params: 8.26M


# **Zoom Baseline Training**

In [ ]:
torch.manual_seed(42)

student_zoom_baseline = GPT2LMHeadModel(zoom_config)
student_zoom_baseline.to("cuda")

zoom_args = TrainingArguments(
    output_dir="./zoom_baseline",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="no",
    report_to="none",
)

zoom_baseline_trainer = Trainer(
    model=student_zoom_baseline,
    args=zoom_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

zoom_baseline_trainer.train()
zoom_baseline_eval = zoom_baseline_trainer.evaluate()
print("Zoom baseline eval loss:", zoom_baseline_eval["eval_loss"])

# **Zoom Distilled Training**

In [ ]:
torch.manual_seed(42)

student_zoom_distilled = GPT2LMHeadModel(zoom_config)
student_zoom_distilled.to("cuda")

zoom_distill_trainer = DistillationTrainer(
    model=student_zoom_distilled,
    teacher_model=teacher,
    temperature=2.0,
    alpha=0.5,
    args=zoom_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

zoom_distill_trainer.train()
zoom_distill_eval = zoom_distill_trainer.evaluate()
print("Zoom distilled eval loss:", zoom_distill_eval["eval_loss"])

In [ ]:
import matplotlib.pyplot as plt

# All data points, sorted by parameter count
sizes_params = [tiny_params, zoom_params, xsmall_params, small_params, medium_params]
gaps = [
    1.340,                          # Tiny
    0.121,                          # Zoom
    (0.101 + 0.107) / 2,            # XSmall (avg of 2 runs)
    (0.089 + 0.096) / 2,            # Small (avg of 2 runs)
    (0.194 + 0.210) / 2,            # Medium (avg of 2 runs)
]
labels = ["Tiny", "Zoom", "XSmall", "Small", "Medium"]

plt.figure(figsize=(8,5.5))
plt.plot(sizes_params, gaps, marker='o', linewidth=2, color="#1f77b4")

for x, y, label in zip(sizes_params, gaps, labels):
    plt.annotate(label, (x, y), textcoords="offset points", xytext=(8, 8), fontsize=9)

plt.xlabel("Student model size (million parameters)")
plt.ylabel("Distillation benefit (baseline loss − distilled loss)")
plt.title("Distillation Benefit vs. Student Model Size")
plt.grid(True, alpha=0.3)
plt.axvspan(7.24, 8.26, alpha=0.15, color="red", label="Sharp collapse region")
plt.legend()
plt.savefig("distillation_gap_full.png", dpi=150, bbox_inches="tight")
plt.show()

# **Tiny Baseline v2**

In [ ]:
torch.manual_seed(789)

student_tiny_baseline_v2 = GPT2LMHeadModel(tiny_config)
student_tiny_baseline_v2.to("cuda")

tiny_args = TrainingArguments(
    output_dir="./tiny_baseline_v2",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="no",
    report_to="none",
)

tiny_baseline_trainer_v2 = Trainer(
    model=student_tiny_baseline_v2,
    args=tiny_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

tiny_baseline_trainer_v2.train()
tiny_baseline_eval_v2 = tiny_baseline_trainer_v2.evaluate()
print("Tiny baseline v2 eval loss:", tiny_baseline_eval_v2["eval_loss"])

# **Tiny Distilled v2**

In [ ]:
torch.manual_seed(789)

student_tiny_distilled_v2 = GPT2LMHeadModel(tiny_config)
student_tiny_distilled_v2.to("cuda")

tiny_distill_trainer_v2 = DistillationTrainer(
    model=student_tiny_distilled_v2,
    teacher_model=teacher,
    temperature=2.0,
    alpha=0.5,
    args=tiny_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

tiny_distill_trainer_v2.train()
tiny_distill_eval_v2 = tiny_distill_trainer_v2.evaluate()
print("Tiny distilled v2 eval loss:", tiny_distill_eval_v2["eval_loss"])

# **Tiny Baseline v3**

In [ ]:
torch.manual_seed(2024)

student_tiny_baseline_v3 = GPT2LMHeadModel(tiny_config)
student_tiny_baseline_v3.to("cuda")

tiny_baseline_trainer_v3 = Trainer(
    model=student_tiny_baseline_v3,
    args=tiny_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

tiny_baseline_trainer_v3.train()
tiny_baseline_eval_v3 = tiny_baseline_trainer_v3.evaluate()
print("Tiny baseline v3 eval loss:", tiny_baseline_eval_v3["eval_loss"])

# **Tiny Distilled v3**

In [ ]:
torch.manual_seed(2024)

student_tiny_distilled_v3 = GPT2LMHeadModel(tiny_config)
student_tiny_distilled_v3.to("cuda")

tiny_distill_trainer_v3 = DistillationTrainer(
    model=student_tiny_distilled_v3,
    teacher_model=teacher,
    temperature=2.0,
    alpha=0.5,
    args=tiny_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

tiny_distill_trainer_v3.train()
tiny_distill_eval_v3 = tiny_distill_trainer_v3.evaluate()
print("Tiny distilled v3 eval loss:", tiny_distill_eval_v3["eval_loss"])

# ***Tiny Alpha Sweep***

In [ ]:
alpha_results = {}

for alpha_val in [0.25, 0.5, 0.75]:
    torch.manual_seed(42)
    m = GPT2LMHeadModel(tiny_config)
    m.to("cuda")
    trainer = DistillationTrainer(
        model=m, teacher_model=teacher, temperature=2.0, alpha=alpha_val,
        args=TrainingArguments(
            output_dir=f"./tiny_alpha_{alpha_val}",
            per_device_train_batch_size=16, per_device_eval_batch_size=16,
            num_train_epochs=1, logging_steps=200, eval_strategy="steps",
            eval_steps=200, save_strategy="no", report_to="none",
        ),
        train_dataset=tokenized_train, eval_dataset=tokenized_val, data_collator=data_collator,
    )
    trainer.train()
    result = trainer.evaluate()
    alpha_results[alpha_val] = result["eval_loss"]
    print(f"Tiny, alpha={alpha_val}: distilled loss = {result['eval_loss']:.3f}")

print(alpha_results)

# **Zoom Alpha Sweep**

In [ ]:
alpha_results_zoom = {}

for alpha_val in [0.25, 0.5, 0.75]:
    torch.manual_seed(42)
    m = GPT2LMHeadModel(zoom_config)
    m.to("cuda")
    trainer = DistillationTrainer(
        model=m, teacher_model=teacher, temperature=2.0, alpha=alpha_val,
        args=TrainingArguments(
            output_dir=f"./zoom_alpha_{alpha_val}",
            per_device_train_batch_size=16, per_device_eval_batch_size=16,
            num_train_epochs=1, logging_steps=200, eval_strategy="steps",
            eval_steps=200, save_strategy="no", report_to="none",
        ),
        train_dataset=tokenized_train, eval_dataset=tokenized_val, data_collator=data_collator,
    )
    trainer.train()
    result = trainer.evaluate()
    alpha_results_zoom[alpha_val] = result["eval_loss"]
    print(f"Zoom, alpha={alpha_val}: distilled loss = {result['eval_loss']:.3f}")

print(alpha_results_zoom)

In [ ]:
torch.manual_seed(999)

student_tiny_a025_v2 = GPT2LMHeadModel(tiny_config)
student_tiny_a025_v2.to("cuda")

trainer = DistillationTrainer(
    model=student_tiny_a025_v2, teacher_model=teacher, temperature=2.0, alpha=0.25,
    args=TrainingArguments(
        output_dir="./tiny_alpha025_v2",
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=1, logging_steps=200, eval_strategy="steps",
        eval_steps=200, save_strategy="no", report_to="none",
    ),
    train_dataset=tokenized_train, eval_dataset=tokenized_val, data_collator=data_collator,
)
trainer.train()
result = trainer.evaluate()
print(f"Tiny, alpha=0.25, seed 999: distilled loss = {result['eval_loss']:.3f}")

In [ ]:
medium_alpha_results = {}
for alpha_val in [0.25, 0.75]:
    torch.manual_seed(42)
    m = GPT2LMHeadModel(medium_config)
    m.to("cuda")
    trainer = DistillationTrainer(
        model=m, teacher_model=teacher, temperature=2.0, alpha=alpha_val,
        args=TrainingArguments(
            output_dir=f"./medium_alpha_{alpha_val}",
            per_device_train_batch_size=16, per_device_eval_batch_size=16,
            num_train_epochs=1, logging_steps=200, eval_strategy="steps",
            eval_steps=200, save_strategy="no", report_to="none",
        ),
        train_dataset=tokenized_train, eval_dataset=tokenized_val, data_collator=data_collator,
    )
    trainer.train()
    result = trainer.evaluate()
    medium_alpha_results[alpha_val] = result["eval_loss"]
    print(f"Medium, alpha={alpha_val}: distilled loss = {result['eval_loss']:.3f}")

print(medium_alpha_results)

# **Zoom, alpha=0.25, second seed**

In [ ]:
torch.manual_seed(999)
student_zoom_a025_v2 = GPT2LMHeadModel(zoom_config)
student_zoom_a025_v2.to("cuda")
trainer = DistillationTrainer(
    model=student_zoom_a025_v2, teacher_model=teacher, temperature=2.0, alpha=0.25,
    args=TrainingArguments(
        output_dir="./zoom_alpha025_v2",
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=1, logging_steps=200, eval_strategy="steps",
        eval_steps=200, save_strategy="no", report_to="none",
    ),
    train_dataset=tokenized_train, eval_dataset=tokenized_val, data_collator=data_collator,
)
trainer.train()
result = trainer.evaluate()
print(f"Zoom, alpha=0.25, seed 999: distilled loss = {result['eval_loss']:.3f}")

# **Medium, alpha=0.25, second seed**

In [ ]:
torch.manual_seed(999)
student_medium_a025_v2 = GPT2LMHeadModel(medium_config)
student_medium_a025_v2.to("cuda")
trainer = DistillationTrainer(
    model=student_medium_a025_v2, teacher_model=teacher, temperature=2.0, alpha=0.25,
    args=TrainingArguments(
        output_dir="./medium_alpha025_v2",
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=1, logging_steps=200, eval_strategy="steps",
        eval_steps=200, save_strategy="no", report_to="none",
    ),
    train_dataset=tokenized_train, eval_dataset=tokenized_val, data_collator=data_collator,
)
trainer.train()
result = trainer.evaluate()
print(f"Medium, alpha=0.25, seed 999: distilled loss = {result['eval_loss']:.3f}")

In [18]:
def train_and_pure_eval(config, alpha_val, seed, temperature=2.0, tag=""):
    torch.manual_seed(seed)
    m = GPT2LMHeadModel(config)
    m.to("cuda")

    args = TrainingArguments(
        output_dir=f"./run_{tag}_{alpha_val}_{seed}",
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=1, logging_steps=200, eval_strategy="no",
        save_strategy="no", report_to="none",
    )

    trainer = DistillationTrainer(
        model=m, teacher_model=teacher, temperature=temperature, alpha=alpha_val,
        args=args, train_dataset=tokenized_train, eval_dataset=tokenized_val,
        data_collator=data_collator,
    )
    trainer.train()

    # Pure CE eval, comparable to baseline
    pure_trainer = Trainer(model=m, args=args, eval_dataset=tokenized_val, data_collator=data_collator)
    pure_ce = pure_trainer.evaluate()["eval_loss"]
    print(f"{tag}, alpha={alpha_val}, seed={seed}: pure CE loss = {pure_ce:.3f}")
    return pure_ce

In [20]:
results = {}
results[("tiny", 0.25)] = train_and_pure_eval(tiny_config, 0.25, 42, tag="tiny")
results[("tiny", 0.5)]  = train_and_pure_eval(tiny_config, 0.5, 42, tag="tiny")
results[("tiny", 0.75)] = train_and_pure_eval(tiny_config, 0.75, 42, tag="tiny")

results[("zoom", 0.25)] = train_and_pure_eval(zoom_config, 0.25, 42, tag="zoom")
results[("zoom", 0.5)]  = train_and_pure_eval(zoom_config, 0.5, 42, tag="zoom")
results[("zoom", 0.75)] = train_and_pure_eval(zoom_config, 0.75, 42, tag="zoom")

results[("medium", 0.25)] = train_and_pure_eval(medium_config, 0.25, 42, tag="medium")
results[("medium", 0.5)]  = train_and_pure_eval(medium_config, 0.5, 42, tag="medium")
results[("medium", 0.75)] = train_and_pure_eval(medium_config, 0.75, 42, tag="medium")

print(results)

Step,Training Loss
200,8.636049
400,7.190143
600,6.383253
800,5.899023
1000,5.606827
1200,5.480803


tiny, alpha=0.25, seed=42: pure CE loss = 6.594


Step,Training Loss
200,9.015433
400,7.571177
600,6.739540
800,6.251436
1000,5.966014
1200,5.837229


tiny, alpha=0.5, seed=42: pure CE loss = 6.251


Step,Training Loss
200,9.345994
400,7.849420
600,6.961346
800,6.453760
1000,6.166049
1200,6.030314


tiny, alpha=0.75, seed=42: pure CE loss = 6.022


... (8 outputs left out)

Step,Training Loss
200,8.071881
400,5.965648
600,5.078785
800,4.679586
1000,4.456224
1200,4.349229


medium, alpha=0.5, seed=42: pure CE loss = 4.740


Step,Training Loss
200,8.301147
400,6.078738
600,5.213862
800,4.848836
1000,4.654001
1200,4.558428


medium, alpha=0.75, seed=42: pure CE loss = 4.533
{('tiny', 0.25): 6.593844890594482, ('tiny', 0.5): 6.2511982917785645, ('tiny', 0.75): 6.0221123695373535, ('zoom', 0.25): 6.278305530548096, ('zoom', 0.5): 5.958295822143555, ('zoom', 0.75): 5.7235002517700195, ('medium', 0.25): 5.094214916229248, ('medium', 0.5): 4.740088939666748, ('medium', 0.75): 4.533179759979248}


In [23]:
core_results = {}
for name, cfg in [("tiny", tiny_config), ("zoom", zoom_config), ("xsmall", xsmall_config),
                   ("small", small_config), ("medium", medium_config)]:
    for seed in [42, 999]:
        core_results[(name, seed)] = train_and_pure_eval(cfg, 0.5, seed, tag=name)

print(core_results)

Step,Training Loss
200,9.015433
400,7.571177
600,6.739540
800,6.251436
1000,5.966014
1200,5.837229


tiny, alpha=0.5, seed=42: pure CE loss = 6.251


Step,Training Loss
200,9.037124
400,7.578655
600,6.747682
800,6.264279
1000,5.982086
1200,5.855129


tiny, alpha=0.5, seed=999: pure CE loss = 6.264


Step,Training Loss
200,8.864537
400,7.301320
600,6.449546
800,5.970270
1000,5.697563
1200,5.576824


zoom, alpha=0.5, seed=42: pure CE loss = 5.958


... (10 outputs left out)

Step,Training Loss
200,8.071881
400,5.965648
600,5.078785
800,4.679586
1000,4.456224
1200,4.349229


medium, alpha=0.5, seed=42: pure CE loss = 4.740


Step,Training Loss
200,8.084410
400,5.977294
600,5.083160
800,4.681301
1000,4.458799
1200,4.351261


medium, alpha=0.5, seed=999: pure CE loss = 4.739
{('tiny', 42): 6.2511982917785645, ('tiny', 999): 6.264448642730713, ('zoom', 42): 5.958295822143555, ('zoom', 999): 5.977873802185059, ('xsmall', 42): 5.785468578338623, ('xsmall', 999): 5.784509658813477, ('small', 42): 5.359232425689697, ('small', 999): 5.353526592254639, ('medium', 42): 4.740088939666748, ('medium', 999): 4.739146709442139}


In [21]:
alpha1_results = {}
for name, cfg in [("tiny", tiny_config), ("zoom", zoom_config), ("medium", medium_config)]:
    alpha1_results[name] = train_and_pure_eval(cfg, 1.0, 42, tag=name)
print(alpha1_results)

Step,Training Loss
200,9.622603
400,7.987570
600,7.002373
800,6.462987
1000,6.173018
1200,6.031732


tiny, alpha=1.0, seed=42: pure CE loss = 5.911


Step,Training Loss
200,9.423000
400,7.635260
600,6.633939
800,6.124757
1000,5.861292
1200,5.734967


zoom, alpha=1.0, seed=42: pure CE loss = 5.612


Step,Training Loss
200,8.441744
400,6.021066
600,5.159441
800,4.826417
1000,4.659211
1200,4.573122


medium, alpha=1.0, seed=42: pure CE loss = 4.465
{'tiny': 5.91141939163208, 'zoom': 5.6123046875, 'medium': 4.46517276763916}


In [1]:
def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
    outputs = model(**inputs)
    student_loss = outputs.loss
    student_logits = outputs.logits
    labels = inputs["labels"]

    with torch.no_grad():
        teacher_logits = self.teacher(**inputs).logits

    T = self.temperature
    mask = (labels != -100).view(-1)  # real tokens only, drop padding

    student_flat = student_logits.view(-1, student_logits.size(-1))[mask]
    teacher_flat = teacher_logits.view(-1, teacher_logits.size(-1))[mask]

    distill_loss = F.kl_div(
        F.log_softmax(student_flat / T, dim=-1),
        F.softmax(teacher_flat / T, dim=-1),
        reduction="batchmean"
    ) * (T ** 2)

    loss = self.alpha * student_loss + (1 - self.alpha) * distill_loss
    return (loss, outputs) if return_outputs else loss

In [20]:
class DistillationTrainerMasked(Trainer):
    def __init__(self, teacher_model=None, temperature=2.0, alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        student_loss = outputs.loss
        student_logits = outputs.logits
        labels = inputs["labels"]

        with torch.no_grad():
            teacher_logits = self.teacher(**inputs).logits

        T = self.temperature
        mask = (labels != -100).view(-1)

        student_flat = student_logits.view(-1, student_logits.size(-1))[mask]
        teacher_flat = teacher_logits.view(-1, teacher_logits.size(-1))[mask]

        distill_loss = F.kl_div(
            F.log_softmax(student_flat / T, dim=-1),
            F.softmax(teacher_flat / T, dim=-1),
            reduction="batchmean"
        ) * (T ** 2)

        loss = self.alpha * student_loss + (1 - self.alpha) * distill_loss
        return (loss, outputs) if return_outputs else loss


def train_and_pure_eval_masked(config, alpha_val, seed, tag=""):
    torch.manual_seed(seed)
    m = GPT2LMHeadModel(config)
    m.to("cuda")
    args = TrainingArguments(
        output_dir=f"./masked_{tag}_{alpha_val}_{seed}",
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=1, logging_steps=200, eval_strategy="no",
        save_strategy="no", report_to="none",
    )
    trainer = DistillationTrainerMasked(
        model=m, teacher_model=teacher, temperature=2.0, alpha=alpha_val,
        args=args, train_dataset=tokenized_train, eval_dataset=tokenized_val,
        data_collator=data_collator,
    )
    trainer.train()
    pure_trainer = Trainer(model=m, args=args, eval_dataset=tokenized_val, data_collator=data_collator)
    ce = pure_trainer.evaluate()["eval_loss"]
    print(f"[MASKED] {tag}, alpha={alpha_val}, seed={seed}: pure CE = {ce:.3f}")
    return ce

masked_results = {}
masked_results[0.25] = train_and_pure_eval_masked(tiny_config, 0.25, 42, tag="tiny")
masked_results[0.5]  = train_and_pure_eval_masked(tiny_config, 0.5, 42, tag="tiny")
print(masked_results)

Step,Training Loss
200,8.632534
400,7.181491
600,6.375366
800,5.892299
1000,5.599626
1200,5.473373


[MASKED] tiny, alpha=0.25, seed=42: pure CE = 6.592


Step,Training Loss
200,9.013827
400,7.565623
600,6.733204
800,6.245950
1000,5.959587
1200,5.830222


[MASKED] tiny, alpha=0.5, seed=42: pure CE = 6.251
{0.25: 6.592096328735352, 0.5: 6.250847816467285}
